# Practical: Tactile Dataset Exploration

2026–2027 Deep Learning Final Coursework · Part A

[Course overview](../README.md) · [Getting started](../docs/GETTING_STARTED.md) · [Project handbook](../resources/2026_2027_DL_Final_Project_Handbook_Shared.pdf) · [Part A guide](../docs/COURSEWORK_GUIDE.md) · [Assessment overview](../docs/ASSESSMENT_OVERVIEW.md) · [Handbook reading guide](../resources/README.md)

## Purpose and learning outcomes

This practical introduces the released data for shape classification (handbook Task 2) and force estimation (handbook Task 3). Both are individual Part A tasks. The archive's legacy names “Task A” and “Task B” refer to these learning tasks; the latter is distinct from the separate Part B group project.

The notebook downloads, verifies and explores 2,000 shape images and 2,000 force images. On completing the practical, students should be able to:

- interpret the image paths, class labels and force-label schema;
- identify the ten shape classes and inspect examples of each;
- explain the relationship between paired press and shear observations;
- use descriptive distributions to inform data preparation and evaluation.

## Assessment context

Part A requires at least three CNN architectures for shape classification and at least two architectures for force estimation, a 3–6 page IEEE report, code and a CID-named model folder. It carries 60 marks and is due on 13 December 2026 at 23:59 UK time (GMT), through Canvas. [Handbook Sections 3–5 and 9](../resources/2026_2027_DL_Final_Project_Handbook_Shared.pdf) define the full requirements. This practical supports dataset exploration; students develop and evaluate their assessed models separately.

Use only the released coursework data for Part A. Neither these data nor the TacVerse GelSight subsets may be used for the separate Part B group project. General-purpose pretrained weights such as ImageNet are allowed for Part A if cited; see [handbook Section 11.3](../resources/2026_2027_DL_Final_Project_Handbook_Shared.pdf#page=41).

Exploration and validation run on CPU. Use a GPU when appropriate for subsequent model training. Before submitting the assessed notebook, restart the runtime, run all cells and save the outputs. This exploration template has no saved outputs.


## 1. Environment and data access

### Google Colab

Open this notebook in Colab and run the setup cell. It clones the coursework repository when necessary and extracts the dataset under `/content`. A temporary Colab runtime may require the data to be downloaded again after the session ends.

### Local Jupyter or VS Code

Clone or download the repository, open this notebook and select a Python environment. The dataset is extracted inside the repository. Install the inspection dependencies in that environment if required:

```bash
python -m pip install numpy pandas pillow matplotlib
```

The download is pinned to release `v1.0.2` (37.44 MB), which distributes the unchanged original dataset. The helper verifies the complete archive's SHA-256 before extraction and preserves existing dataset folders.


In [ ]:
from pathlib import Path
import subprocess
import sys

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

REPO_NAME = "2026-2027-Deep-Learning-Final-Coursework-Part-A"
REPO_URL = f"https://github.com/dianzdd123/{REPO_NAME}.git"

# Find an existing checkout, whether this notebook starts in the root or notebooks/.
candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / REPO_NAME]
REPO = next((p for p in candidates if (p / "scripts/download_dataset.py").is_file()), None)
if REPO is None:
    REPO = Path.cwd() / REPO_NAME
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)

# Colab uses /content; a local session uses the checkout directory.
DATA_PARENT = Path("/content") if Path("/content").is_dir() else REPO
ROOT = DATA_PARENT / "Tactile_DL_2027"
if not ROOT.exists():
    subprocess.run(
        [sys.executable, str(REPO / "scripts/download_dataset.py"), "--output-dir", str(DATA_PARENT)],
        check=True,
    )

# Re-check an existing extraction as well as a fresh download.
subprocess.run(
    [sys.executable, str(REPO / "scripts/verify_dataset.py"), "--data", str(ROOT)],
    check=True,
)
print("Dataset location:", ROOT)


### Use an existing dataset archive

To prepare a local copy from an existing archive, run the following command in a terminal before continuing with the notebook:

```bash
python scripts/download_dataset.py --archive /path/to/Tactile_DL_2027.zip
```

The helper also accepts the uploaded filename `Tactile_DL_2027 (1).zip`. Enclose paths containing spaces in quotation marks. The archive contents and checksum identify the release independently of its filename.


## 2. Label schemas

Each `filename` is relative to its task folder. Resolve a shape filename beneath `ROOT / "shape_classification"` and a force filename beneath `ROOT / "force_regression"`.

Shape classification has one integer target, `class_id`. Force estimation has three targets in newtons: `fx`, `fy` and `fz`. The `contact_id` field links a normal press to its subsequent shear image. The `is_shear` field supports exploration and evaluation; it is unavailable as a model input during hidden-test inference.


In [ ]:
shape_root = ROOT / "shape_classification"
force_root = ROOT / "force_regression"
shape = pd.read_csv(shape_root / "labels.csv")
classes = pd.read_csv(shape_root / "classes.csv").sort_values("class_id")
force = pd.read_csv(force_root / "labels.csv")

print(f"Shape images: {len(shape):,}")
print(f"Force images: {len(force):,}; physical contacts: {force.contact_id.nunique():,}")
display(shape.head())
display(force.head())


## 3. Shape classes and image examples

Use the supplied class IDs consistently throughout preprocessing, training and inference. Retain the original label mapping when sorting or displaying the data. The released dataset contains 200 images per class.

The figure below displays the first labelled image in each class. These examples illustrate tactile camera observations of contact; their selection does not rank the classes by representativeness or difficulty.


In [ ]:
class_counts = shape.groupby(["class_id", "class_name"]).size().rename("images").reset_index()
display(class_counts)
assert len(classes) == 10
assert class_counts.images.eq(200).all()

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for ax, label in zip(axes.flat, classes.itertuples(index=False)):
    row = shape.loc[shape.class_id.eq(label.class_id)].iloc[0]
    with Image.open(shape_root / row.filename) as image:
        ax.imshow(image)
    ax.set_title(f"{label.class_id}: {label.class_name}")
    ax.axis("off")
fig.suptitle("One released tactile image for each shape class")
plt.tight_layout()
plt.show()


In [ ]:
# Images are 320 pixels wide and 240 pixels high.
# NumPy reports array dimensions as (height, width, colour channels).
example_path = shape_root / shape.iloc[0].filename
with Image.open(example_path) as image:
    pixels = np.asarray(image.convert("RGB"))
    print("Pillow size (width, height):", image.size)
print("NumPy shape (height, width, channels):", pixels.shape)
print("Stored pixel type:", pixels.dtype)


## 4. Paired force observations

Each `contact_id` has two images: `is_shear = 0` identifies the normal press, and `is_shear = 1` identifies the observation after lateral displacement. Keep both images in the same data split to prevent information about one contact from crossing between training, validation and test subsets.

The targets `fx` and `fy` are tangential force components; `fz` is normal force. Under the supplied sign convention, negative `fz` indicates pressing into the sensor. Hidden-test filenames are anonymous, so force estimates must be based on image content alone.


In [ ]:
contact = int(force.contact_id.min())
pair = force.loc[force.contact_id.eq(contact)].sort_values("is_shear")
assert len(pair) == 2 and set(pair.is_shear) == {0, 1}
display(pair)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, row in zip(axes, pair.itertuples(index=False)):
    with Image.open(force_root / row.filename) as image:
        ax.imshow(image)
    kind = "Normal press" if row.is_shear == 0 else "After shear"
    ax.set_title(f"{kind} — contact {row.contact_id}\nfx={row.fx:.3f}, fy={row.fy:.3f}, fz={row.fz:.3f} N")
    ax.axis("off")
plt.tight_layout()
plt.show()


## 5. Force distributions

The tables below describe the released subset. Estimate any means and standard deviations used for normalisation from the **training split only**, then apply those fixed values to validation and test data. The full-release statistics are descriptive and must not be used as fitted preprocessing parameters. Convert predictions back to newtons before reporting errors.

Compare the tangential spread of press and shear observations. Half of the observations are normal presses, so a pooled score may conceal weak shear estimation. The coursework requires separate press/shear results and comparison with a training-mean baseline.


In [ ]:
force_axes = ["fx", "fy", "fz"]
print("All released observations; standard deviation is sample SD (ddof=1):")
display(force[force_axes].describe().loc[["count", "mean", "std", "min", "max"]])

print("Press (0) versus shear (1), in newtons:")
display(force.groupby("is_shear")[force_axes].agg(["mean", "std"]))

# Every contact should contribute exactly one image of each kind.
pair_counts = force.groupby(["contact_id", "is_shear"]).size().unstack()
assert pair_counts.shape == (1000, 2)
assert pair_counts.eq(1).all().all()
print("Verified: 1,000 complete press/shear pairs.")


## 6. Experimental design

[Handbook Section 3.3](../resources/2026_2027_DL_Final_Project_Handbook_Shared.pdf#page=11) requires 60% training, 20% validation and 20% test:

| Task | Split unit | Training | Validation | Test |
|---|---|---:|---:|---:|
| Shape classification | Images, stratified by class | 1,200 images | 400 images | 400 images |
| Force estimation | Whole `contact_id` groups | 600 contacts / 1,200 images | 200 contacts / 400 images | 200 contacts / 400 images |

Fix and record the random seed. Shape stratification gives 120/40/40 images per class. For force estimation, verify that the contact-ID sets do not overlap. Use validation data for model selection and reserve the test split for final evaluation.

Address the following considerations when designing the experiments:

- Image resolution: assess the contact details lost through downsampling and justify the input size.
- Augmentation validity: select transformations that preserve task labels. Force-image rotations or flips may require corresponding changes to vector labels; justify the image-to-force coordinate mapping.
- Classification errors: use a confusion matrix to examine class-specific behaviour alongside aggregate accuracy.
- Force baselines: compare the model with the training-set mean on the shear subset, with particular attention to `fx` and `fy`.

The archive contains `example_submission/predict.py`, example checkpoints and the script used to produce them. These files demonstrate the required inference interface. The supplied checkpoints were trained on all released images and cannot establish held-out coursework performance. Continue with the [Part A guide](../docs/COURSEWORK_GUIDE.md) and [submission interface reference](../example_submission/README.md) when preparing assessed experiments and the model folder.
